# Day 6 — Venture Knowledge Base & Semantic Retrieval (Layers 2 & 3)

Historical memory for the platform: embed past campaigns (SBERT), store in a FAISS vector
DB, and retrieve similar campaigns + real outcomes + benchmarks as **evidence** that sits
next to the Day-4 prediction. Logic lives in `src/knowledge_base/` and `src/retrieval/`;
this notebook imports those modules and runs them end to end.

**Honesty:** retrieval cites only real features (title, category, country, goal, outcome).
Founder / market / valuation retrieval is **deferred** — no such data exists.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import numpy as np, pandas as pd

# ONE warm stack. Building the pipeline loads the Day-4 predictor (lightgbm) BEFORE the
# FAISS stack — the required OpenMP load order (see src/retrieval/__init__.py + pipeline.py).
# We reuse its components across every phase below.
from src.retrieval.pipeline import VentureIntelligencePipeline
pipe = VentureIntelligencePipeline()
evidence = pipe.evidence
retriever = evidence.retriever
search = retriever.engine
vdb = search.vector_db
print("Warm stack ready. Predictor attached:", pipe.predictor is not None,
      "| FAISS vectors:", vdb.n_vectors)

Warm stack ready. Predictor attached: True | FAISS vectors: 265171


## Phase A — Understanding Semantic Retrieval

Keyword search fails on 5.6-word titles: a *same-meaning* pair with no shared words scored
0.0 on keyword overlap but 0.34 on SBERT cosine, while a pair sharing the word "board" but
meaning different things scored *higher* on keywords — keyword ranks them backwards. SBERT
maps each campaign to a 384-dim dense vector where cosine = meaning; FAISS searches it
exactly (no ANN needed at 265k). Full notes: `docs/day6_phaseA_semantic_retrieval_notes.md`.

## Phase B — Venture Knowledge Base (train-only)

`VentureKnowledgeBase` turns the **train** campaigns into a metadata table + a serialized
venture-record string per campaign (title + category + country + goal band). Train-only keeps
retrieval evidence non-circular. Reading the persisted KB:

In [2]:
md_table = pd.read_csv("artifacts/embeddings/metadata.csv")
manifest = json.load(open("artifacts/embeddings/metadata.json"))
print("KB records:", manifest["n_records"], "| split:", manifest["split"],
      "| success rate:", manifest["success_rate"])
for t in md_table["text"].head(3): print(" -", t)

KB records: 265171 | split: train | success rate: 0.4038
 - A Winter Storm - A Novel. A Fiction project in the Publishing category, based in the United States, seeking a modest goal of about $6,000 (launched 2012).
 - PilaFoot. A Technology project in the Technology category, based in Belgium, seeking a large goal of about $44,674 (launched 2017).
 - JB Couture Cupcakes Hits the Shore!. A Food project in the Food category, based in the United States, seeking a large goal of about $28,000 (launched 2014).


## Phase C — Embedding Generation (SBERT)

`EmbeddingGenerator` encodes each record with `all-MiniLM-L6-v2` into an L2-normalized
384-dim vector (so cosine == dot product). Persisted as `startup_embeddings.npy` +
`embedding_model.json` (descriptor, not a pickled model).

In [3]:
emb = np.load("artifacts/embeddings/startup_embeddings.npy", mmap_mode="r")
info = json.load(open("artifacts/embeddings/embedding_model.json"))
print("embeddings:", emb.shape, info["dtype"], "| model:", info["model_name"])
print("unit norm (row 0):", round(float(np.linalg.norm(emb[0])), 4))

embeddings: (265171, 384) float32 | model: all-MiniLM-L6-v2
unit norm (row 0): 1.0


## Phase D — Vector Database (FAISS exact cosine)

`VectorDatabase` = FAISS `IndexFlatIP` over normalized vectors (exact cosine) + a
row→campaign mapping. Exact, not approximate: verified below against a numpy brute-force.

In [4]:
q = emb[13].astype("float32")
sims, rows = vdb.search(q, k=4)
bf = np.argsort(np.asarray(emb) @ q)[::-1][:4]
print("FAISS rows :", rows[0].tolist())
print("bruteforce :", bf.tolist(), "| exact match:", rows[0].tolist() == bf.tolist())

FAISS rows : [13, 61613, 84777, 95542]
bruteforce : [13, 61613, 84777, 95542] | exact match: True


## Phase E — Semantic Search Engine

Embed a query the same way as the corpus, FAISS top-k, self-exclusion + real-facet filters.

In [5]:
query = {"name": "Handcrafted wooden board game for families", "category": "Tabletop Games",
         "main_category": "Games", "country": "US", "currency": "USD",
         "usd_goal_real": 8000, "goal": 8000, "launched": "2016-05-01"}
print("SIMILAR (any outcome):")
for n in search.search(query, k=4):
    print("  %.3f | %-28s | %s" % (n["similarity"], str(n["title"])[:28], n["outcome"]))
print("SIMILAR SUCCESSFUL (filtered):")
for n in search.search(query, k=4, filters={"outcome": "successful"}):
    print("  %.3f | %-28s | %s" % (n["similarity"], str(n["title"])[:28], n["outcome"]))

SIMILAR (any outcome):


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0.886 | Hand Made Wooden Gaming Tabl | failed
  0.862 | Handcrafted Hardwood Heirloo | successful
  0.858 | Board Game                   | failed
  0.855 | Boardcrafting                | successful
SIMILAR SUCCESSFUL (filtered):
  0.862 | Handcrafted Hardwood Heirloo | successful
  0.855 | Boardcrafting                | successful
  0.845 | Handcrafted Wooden Deck Box  | successful
  0.830 | LOAD the Board Game          | successful


## Phase F — Historical Venture Retriever

One search feeds similar / successful / failed lists + a funding benchmark (k-NN prior +
goal split) + a whole-KB category benchmark. Founder/market/valuation benchmarks deferred.

In [6]:
bundle = retriever.retrieve(query)
fb, cb = bundle["funding_benchmark"], bundle["category_benchmark"]
print("k-NN prior (funding rate of %d neighbors): %.3f" % (fb["n_neighbors"], fb["success_rate"]))
print("  winners median goal $%s vs losers $%s | query $%s -> %s the winners' median"
      % (fb["successful"]["median_goal_usd"], fb["failed"]["median_goal_usd"],
         fb["query_goal_usd"], fb["query_goal_vs_successful_median"]))
print("  %s category: %.0f%% fund overall" % (cb["main_category"], 100*cb["success_rate"]))
print("  deferred:", list(bundle["deferred_benchmarks"].keys()))

k-NN prior (funding rate of 50 neighbors): 0.340
  winners median goal $4000.0 vs losers $9000.0 | query $8000.0 -> above the winners' median
  Games category: 44% fund overall
  deferred: ['founder_benchmark', 'market_benchmark', 'valuation_benchmark', 'exit_benchmark']


## Phase G — Retrieval Evidence Generator

Distill the bundle into a readable evidence pack; compare the retrieval prior to a model
probability (corroborates / more optimistic / more cautious).

In [7]:
pack = evidence.generate(query, model_probability=0.28)
print(pack["evidence_text"])
print("\nprior_vs_model @0.28:", pack["prior_vs_model"])

Retrieved 50 similar past campaigns: 17 funded (34%) — a retrieval-based estimate of this campaign's funding chance. Similar campaigns that funded typically sought a small goal (median $4,000); those that failed sought a modest goal (median $9,000). This campaign's $8,000 goal is above the funded median. Across the Games category overall, 44% of campaigns fund. The prediction model estimates 28%; retrieved evidence corroborates the model. Note: Historical outcomes of similar past campaigns — context for, not proof of, this campaign's result. Based on campaign metadata and titles only (no founder / market / financial data).

prior_vs_model @0.28: corroborates the model


## Phase H — Venture Intelligence Pipeline (prediction + evidence)

`analyze(raw)` returns the Day-4 calibrated prediction **and** retrieved evidence together —
non-circular (KB is train-only). We query an unseen **TEST** campaign.

In [8]:
test_idx = np.load("artifacts/splits/test_index.npy")
raw = pd.read_csv("data/processed/kickstarter_processed.csv").iloc[test_idx]\
        .sample(1, random_state=3).iloc[0].to_dict()
rep = pipe.analyze(raw)
print("Campaign:", str(rep.query["name"])[:44], "|", rep.query["category"])
if rep.prediction:
    print("  MODEL:     p=%.3f -> %s" % (rep.prediction["funding_probability"], rep.prediction["predicted_label"]))
print("  RETRIEVAL: prior=%.3f -> %s" % (rep.evidence["retrieval_prior"], rep.evidence["prior_vs_model"]))
print("  EVIDENCE:", rep.evidence["evidence_text"][:220], "...")

Campaign: (Thanksgiving Challenge) Thanksgiving (Thank | Hip-Hop
  MODEL:     p=0.178 -> failed
  RETRIEVAL: prior=0.340 -> is more optimistic than the model
  EVIDENCE: Retrieved 50 similar past campaigns: 17 funded (34%) — a retrieval-based estimate of this campaign's funding chance. Similar campaigns that funded typically sought a small goal (median $3,500); those that failed sought a ...


## Phase I — Knowledge Base Serialization (versioned + verifiable)

`KnowledgeBaseSerializer` hashes every artifact (SHA-256) into one manifest; `verify()`
re-hashes to detect drift. Content-hash = version identity.

In [9]:
from src.retrieval.serialization import KnowledgeBaseSerializer
ser = KnowledgeBaseSerializer().build(); ser.save()
print(ser.describe())
v = ser.verify()
print("verify all_ok:", v["all_ok"], "|", {k: ("ok" if ok else "MISMATCH") for k, ok in v["artifacts"].items()})

Venture KB v1.0.0: 265171 train campaigns, all-MiniLM-L6-v2 (384-dim), FAISS flat_ip (265171 vectors, cosine (inner product on L2-normalized vectors)). 5 artifacts hashed. Deferred: 3 capabilities (no data).


verify all_ok: True | {'embeddings': 'ok', 'faiss_index': 'ok', 'metadata_csv': 'ok', 'mapping': 'ok', 'embedding_model_json': 'ok'}


## Day 6 complete — Layers 2 & 3 ✅

Train-only KB → SBERT embeddings → FAISS exact cosine → semantic search → venture retriever
(similar / success / failure / benchmarks + k-NN prior) → evidence generator (prior-vs-model)
→ intelligence pipeline (prediction + evidence) → versioned serialization. Every result cites
only real features; founder / market / valuation retrieval is deferred, never fabricated.